# Milestone 1 - Product-Level Tabular MLP

Goal: predict a product-level `rating_band` from tabular product metadata using a Keras/TensorFlow MLP.

This notebook uses product-level rows, not review-level rows, for the main model. It preserves the product-level split from Milestone 0 and compares the MLP honestly against simple baselines.


## 0. Setup


In [1]:
from __future__ import annotations

import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

PRODUCT_LEVEL_PATH = PROCESSED_DIR / "product_level.parquet"
PRODUCT_TRAIN_PATH = PROCESSED_DIR / "product_train.parquet"
PRODUCT_VAL_PATH = PROCESSED_DIR / "product_val.parquet"
PRODUCT_TEST_PATH = PROCESSED_DIR / "product_test.parquet"

REVIEW_TRAIN_PATH = PROCESSED_DIR / "train.parquet"
REVIEW_VAL_PATH = PROCESSED_DIR / "val.parquet"
REVIEW_TEST_PATH = PROCESSED_DIR / "test.parquet"

KERAS_MODEL_PATH = MODELS_DIR / "tabular_mlp.keras"
PREPROCESSOR_PATH = MODELS_DIR / "tabular_mlp_preprocessor.joblib"
LABEL_ENCODER_PATH = MODELS_DIR / "tabular_mlp_label_encoder.joblib"
METRICS_PATH = REPORTS_DIR / "milestone1_tabular_mlp_metrics.csv"
MODEL_COMPARISON_PATH = REPORTS_DIR / "tabular_mlp_model_comparison.csv"
CLASSIFICATION_REPORT_PATH = REPORTS_DIR / "milestone1_tabular_mlp_classification_report.csv"
VAL_PREDICTIONS_PATH = PROCESSED_DIR / "tabular_mlp_predictions_val.csv"
TEST_PREDICTIONS_PATH = PROCESSED_DIR / "tabular_mlp_predictions_test.csv"

TARGET = "rating_band"
TARGET_SOURCE = "target_rating"
BAND_ORDER = ["low", "medium", "high"]


## 1. Load Product-Level Data

Preferred input is `product_train.parquet`, `product_val.parquet`, and `product_test.parquet`. If those are not available, the notebook loads `product_level.parquet` and attaches the frozen product split from Milestone 0 review-level split files.


In [3]:
def load_product_splits() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    if PRODUCT_TRAIN_PATH.exists() and PRODUCT_VAL_PATH.exists() and PRODUCT_TEST_PATH.exists():
        return (
            pd.read_parquet(PRODUCT_TRAIN_PATH),
            pd.read_parquet(PRODUCT_VAL_PATH),
            pd.read_parquet(PRODUCT_TEST_PATH),
            "product split parquet files",
        )

    product_level = pd.read_parquet(PRODUCT_LEVEL_PATH)
    if "split" in product_level.columns:
        return (
            product_level.query("split == 'train'").copy(),
            product_level.query("split == 'val'").copy(),
            product_level.query("split == 'test'").copy(),
            "split column in product_level.parquet",
        )

    train_products = pd.read_parquet(REVIEW_TRAIN_PATH)[["product_id"]].drop_duplicates()
    val_products = pd.read_parquet(REVIEW_VAL_PATH)[["product_id"]].drop_duplicates()
    test_products = pd.read_parquet(REVIEW_TEST_PATH)[["product_id"]].drop_duplicates()
    train_products["split"] = "train"
    val_products["split"] = "val"
    test_products["split"] = "test"
    split_map = pd.concat([train_products, val_products, test_products], ignore_index=True)

    product_level = product_level.merge(split_map, on="product_id", how="inner", validate="one_to_one")
    return (
        product_level.query("split == 'train'").copy(),
        product_level.query("split == 'val'").copy(),
        product_level.query("split == 'test'").copy(),
        "product_level.parquet joined to frozen Milestone 0 product split",
    )

product_train_df, product_val_df, product_test_df, split_source = load_product_splits()

print("Split source:", split_source)
print("Product train:", product_train_df.shape)
print("Product val:", product_val_df.shape)
print("Product test:", product_test_df.shape)
display(product_train_df.head(3))


Split source: product_level.parquet joined to frozen Milestone 0 product split
Product train: (17759, 16)
Product val: (3806, 16)
Product test: (3806, 16)


,product_id,product_title,store_name,price_num,average_rating,rating_number,primary_image_url,cached_image_path,image_cache_status,main_category_clean,review_count,review_rating_mean,review_rating_median,negative_review_count,positive_review_count,split
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)",Howard Products,NaN,4.8,10,https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg,data/images/all_beauty/B01CUPMQZE_95d904a0ec.jpg,cached,All Beauty,1,5.0,5.0,0,1,train
1,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Imitation Eyebrow Tattoos, 4D Hair-like Authentic Eyebrows Waterproo...",Cherioll,NaN,3.1,102,https://m.media-amazon.com/images/I/71GJhXQGvyL._SL1500_.jpg,data/images/all_beauty/B088FKY3VD_de33cbc7ba.jpg,cached,All Beauty,2,3.0,3.0,1,1,train
3,B07CV988JX,TOODOO 8 Sets Mermaid Face Gems Glitter Sticker Rhinestone Bindis Crystal Face Tattoo Forehead Decorations for Women...,TOODOO,NaN,4.1,13,https://m.media-amazon.com/images/I/81lMslATIzL._SL1500_.jpg,data/images/all_beauty/B07CV988JX_3621c25233.jpg,cached,All Beauty,1,5.0,5.0,0,1,train


## 2. Product-Level Leakage Check

Train, validation, and test product IDs must have zero overlap.


In [4]:
train_products = set(product_train_df["product_id"].dropna())
val_products = set(product_val_df["product_id"].dropna())
test_products = set(product_test_df["product_id"].dropna())

overlap_report = {
    "train_val_overlap": len(train_products & val_products),
    "train_test_overlap": len(train_products & test_products),
    "val_test_overlap": len(val_products & test_products),
}
leakage_pass = all(value == 0 for value in overlap_report.values())

display(pd.DataFrame([overlap_report]))
print("Leakage check:", "PASS" if leakage_pass else "FAIL")
if not leakage_pass:
    raise ValueError("Product leakage detected across product-level splits.")


,train_val_overlap,train_test_overlap,val_test_overlap
0,0,0,0


Leakage check: PASS


## 3. Create Product-Level Target

`rating_band` is created from `review_rating_mean` when a product has reviews in the sample; otherwise it falls back to metadata `average_rating`. The raw target source columns are excluded from model inputs.

Default bins:
- low: rating < 3.5
- medium: 3.5 <= rating < 4.5
- high: rating >= 4.5

If a class is too small, the notebook switches to quantile-based bins and documents that choice.


In [5]:
def add_rating_target(df: pd.DataFrame, quantile_bins: bool = False) -> pd.DataFrame:
    out = df.copy()
    review_rating = pd.to_numeric(out.get("review_rating_mean"), errors="coerce")
    metadata_rating = pd.to_numeric(out.get("average_rating"), errors="coerce")
    out[TARGET_SOURCE] = review_rating.fillna(metadata_rating)
    if quantile_bins:
        out[TARGET] = pd.qcut(out[TARGET_SOURCE], q=3, labels=BAND_ORDER, duplicates="drop")
    else:
        out[TARGET] = pd.cut(
            out[TARGET_SOURCE],
            bins=[-np.inf, 3.5, 4.5, np.inf],
            labels=BAND_ORDER,
            right=False,
        )
    return out

combined_for_target = pd.concat([
    product_train_df.assign(_split="train"),
    product_val_df.assign(_split="val"),
    product_test_df.assign(_split="test"),
], ignore_index=True)
combined_for_target = add_rating_target(combined_for_target, quantile_bins=False)

train_counts = combined_for_target.query("_split == 'train'")[TARGET].value_counts().reindex(BAND_ORDER, fill_value=0)
min_class_share = train_counts.min() / train_counts.sum()
use_quantile_bins = bool(min_class_share < 0.05)

if use_quantile_bins:
    combined_for_target = add_rating_target(combined_for_target, quantile_bins=True)
    target_strategy = "quantile-based bins because fixed bins were too imbalanced"
else:
    target_strategy = "fixed rating bands: low < 3.5, medium 3.5-4.5, high >= 4.5"

combined_for_target = combined_for_target.dropna(subset=[TARGET, TARGET_SOURCE]).copy()
product_train_df = combined_for_target.query("_split == 'train'").drop(columns=["_split"]).copy()
product_val_df = combined_for_target.query("_split == 'val'").drop(columns=["_split"]).copy()
product_test_df = combined_for_target.query("_split == 'test'").drop(columns=["_split"]).copy()

print("Target strategy:", target_strategy)

def target_distribution(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = df[TARGET].astype(str).value_counts().reindex(BAND_ORDER, fill_value=0)
    out = counts.rename_axis("rating_band").reset_index(name="count")
    out["split"] = split_name
    out["pct"] = (out["count"] / out["count"].sum() * 100).round(1)
    return out[["split", "rating_band", "count", "pct"]]

target_dist = pd.concat([
    target_distribution(product_train_df, "train"),
    target_distribution(product_val_df, "val"),
    target_distribution(product_test_df, "test"),
], ignore_index=True)

display(target_dist)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=target_dist, x="rating_band", y="pct", hue="split", order=BAND_ORDER, ax=ax)
ax.set_title("Product rating band distribution by split")
ax.set_ylabel("Percent")
ax.set_xlabel("Rating band")
plt.show()


Target strategy: fixed rating bands: low < 3.5, medium 3.5-4.5, high >= 4.5


,split,rating_band,count,pct
0,train,low,5254,29.6
1,train,medium,2956,16.6
2,train,high,9549,53.8
3,val,low,1149,30.2
4,val,medium,646,17.0
5,val,high,2011,52.8
6,test,low,1157,30.4
7,test,medium,640,16.8
8,test,high,2009,52.8


/var/folders/r_/zff9yrrx0nx1qlptw7y12jy80000gn/T/ipykernel_33393/3263558848.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
product_train_df.to_parquet(PRODUCT_TRAIN_PATH, index=False)
product_val_df.to_parquet(PRODUCT_VAL_PATH, index=False)
product_test_df.to_parquet(PRODUCT_TEST_PATH, index=False)

print("Saved product-level frozen splits:")
for path in [PRODUCT_TRAIN_PATH, PRODUCT_VAL_PATH, PRODUCT_TEST_PATH]:
    print("-", path.relative_to(PROJECT_ROOT))


Saved product-level frozen splits:
- data/processed/product_train.parquet
- data/processed/product_val.parquet
- data/processed/product_test.parquet


## 4. Non-Leaking Tabular Features

Allowed features are metadata-derived numeric/categorical features. We exclude raw target columns and identifiers: `rating`, `rating_num`, `sentiment_label`, `average_rating`, `review_rating_mean`, `rating_band`, `product_id`, raw review text, and raw product title text.


In [7]:
def add_tabular_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    title = out.get("product_title", pd.Series(index=out.index, dtype="object")).fillna("").astype(str)
    description = out.get("description", pd.Series(index=out.index, dtype="object"))
    features = out.get("features", pd.Series(index=out.index, dtype="object"))

    out["title_length"] = title.str.len()
    out["title_word_count"] = title.str.split().str.len()
    out["description_length"] = description.fillna("").astype(str).str.len()
    out["feature_count"] = features.map(lambda value: len(value) if isinstance(value, (list, tuple, np.ndarray)) else 0)
    out["has_price"] = pd.to_numeric(out.get("price_num", pd.Series(index=out.index, dtype="float")), errors="coerce").notna().astype(int)
    out["log_price"] = np.log1p(pd.to_numeric(out.get("price_num", pd.Series(index=out.index, dtype="float")), errors="coerce"))
    out["log_rating_number"] = np.log1p(pd.to_numeric(out.get("rating_number", pd.Series(index=out.index, dtype="float")), errors="coerce"))
    out["has_image_url"] = out.get("primary_image_url", pd.Series(index=out.index, dtype="object")).notna().astype(int)
    out["has_cached_image"] = out.get("cached_image_path", pd.Series(index=out.index, dtype="object")).fillna("").ne("").astype(int)
    return out

train_model_df = add_tabular_features(product_train_df)
val_model_df = add_tabular_features(product_val_df)
test_model_df = add_tabular_features(product_test_df)

numeric_features = [
    "price_num",
    "log_price",
    "rating_number",
    "log_rating_number",
    "has_price",
    "has_image_url",
    "has_cached_image",
    "title_length",
    "title_word_count",
    "description_length",
    "feature_count",
    "review_count",
]

categorical_features = [
    "store_name",
    "main_category_clean",
    "image_cache_status",
]

numeric_features = [col for col in numeric_features if col in train_model_df.columns]
categorical_features = [col for col in categorical_features if col in train_model_df.columns]
feature_columns = numeric_features + categorical_features

excluded_leakage_columns = [
    "rating", "rating_num", "sentiment_label", "average_rating", "review_rating_mean",
    "review_rating_median", "negative_review_count", "positive_review_count",
    TARGET_SOURCE, TARGET, "product_id", "review_text", "clean_review_text", "product_title",
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Excluded leakage/raw text columns:", excluded_leakage_columns)


Numeric features: ['price_num', 'log_price', 'rating_number', 'log_rating_number', 'has_price', 'has_image_url', 'has_cached_image', 'title_length', 'title_word_count', 'description_length', 'feature_count', 'review_count']
Categorical features: ['store_name', 'main_category_clean', 'image_cache_status']
Excluded leakage/raw text columns: ['rating', 'rating_num', 'sentiment_label', 'average_rating', 'review_rating_mean', 'review_rating_median', 'negative_review_count', 'positive_review_count', 'target_rating', 'rating_band', 'product_id', 'review_text', 'clean_review_text', 'product_title']


In [8]:
X_train = train_model_df[feature_columns]
X_val = val_model_df[feature_columns]
X_test = test_model_df[feature_columns]

y_train = train_model_df[TARGET].astype(str)
y_val = val_model_df[TARGET].astype(str)
y_test = test_model_df[TARGET].astype(str)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
display(X_train.head())


X_train: (17759, 15)
X_val: (3806, 15)
X_test: (3806, 15)


,price_num,log_price,rating_number,log_rating_number,has_price,has_image_url,has_cached_image,title_length,title_word_count,description_length,feature_count,review_count,store_name,main_category_clean,image_cache_status
0,NaN,NaN,10,2.397895,0,1,1,51,6,0,0,1,Howard Products,All Beauty,cached
1,NaN,NaN,102,4.634729,0,1,1,158,22,0,0,2,Cherioll,All Beauty,cached
2,NaN,NaN,13,2.639057,0,1,1,142,20,0,0,1,TOODOO,All Beauty,cached
3,NaN,NaN,10,2.397895,0,1,1,134,20,0,0,1,Yun Mei Hair,All Beauty,cached
4,NaN,NaN,21,3.091042,0,1,1,92,17,0,0,53,BORELTH,All Beauty,cached


## 5. Preprocessing

Numeric features are imputed and scaled. Categorical features are imputed and one-hot encoded.


In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=20, sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

X_train_processed = np.nan_to_num(preprocessor.fit_transform(X_train), nan=0.0, posinf=0.0, neginf=0.0).astype("float32")
X_val_processed = np.nan_to_num(preprocessor.transform(X_val), nan=0.0, posinf=0.0, neginf=0.0).astype("float32")
X_test_processed = np.nan_to_num(preprocessor.transform(X_test), nan=0.0, posinf=0.0, neginf=0.0).astype("float32")

label_encoder = LabelEncoder()
label_encoder.fit(BAND_ORDER)
y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Processed train shape:", X_train_processed.shape)
print("Classes:", list(label_encoder.classes_))


Processed train shape: (17759, 36)
Classes: [np.str_('high'), np.str_('low'), np.str_('medium')]


## 6. Baselines

Majority-class and logistic-regression baselines are trained before the neural model.


In [10]:
def evaluate_predictions(model_name: str, split_name: str, y_true, y_pred) -> dict:
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

metrics = []

majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(X_train_processed, y_train)
for split_name, X_eval, y_eval in [("val", X_val_processed, y_val), ("test", X_test_processed, y_test)]:
    pred = majority_model.predict(X_eval)
    metrics.append(evaluate_predictions("majority_baseline", split_name, y_eval, pred))

logistic_model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)
logistic_model.fit(X_train_processed, y_train)
for split_name, X_eval, y_eval in [("val", X_val_processed, y_val), ("test", X_test_processed, y_test)]:
    pred = logistic_model.predict(X_eval)
    metrics.append(evaluate_predictions("logistic_regression_baseline", split_name, y_eval, pred))

display(pd.DataFrame(metrics).round(4))


,model,split,accuracy,macro_f1,weighted_f1
0,majority_baseline,val,0.5284,0.2305,0.3653
1,majority_baseline,test,0.5279,0.2303,0.3647
2,logistic_regression_baseline,val,0.4627,0.4405,0.4719
3,logistic_regression_baseline,test,0.4506,0.4292,0.4604


## 7. Keras / TensorFlow Tabular MLP

The MLP trains on the full processed product-level training set. Class imbalance is handled with `class_weight`, not by downsampling the training set. Early stopping monitors validation loss and restores the best weights.


In [11]:
class_values = np.unique(y_train_encoded)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_values,
    y=y_train_encoded,
)
class_weight = {int(cls): float(weight) for cls, weight in zip(class_values, class_weights_array)}
class_weight


{0: 0.6199252975878801, 1: 1.126697119654866, 2: 2.002593594948128}

In [12]:
def build_mlp(input_dim: int, num_classes: int) -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.20),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
        run_eagerly=True,
    )
    return model

keras_mlp = build_mlp(X_train_processed.shape[1], len(label_encoder.classes_))
keras_mlp.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 643 (2.51 KB)

 Trainable params: 643 (2.51 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
class_values = np.unique(y_train_encoded)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_values,
    y=y_train_encoded,
)
class_weight = {int(cls): float(weight) for cls, weight in zip(class_values, class_weights_array)}

print("Keras training rows:", X_train_processed.shape[0])
print("Keras training columns:", X_train_processed.shape[1])
print("Class weights:", class_weight)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = keras_mlp.fit(
    X_train_processed,
    y_train_encoded,
    validation_data=(X_val_processed, y_val_encoded),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1,
)

print("Epochs trained:", len(history.history["loss"]))


Keras training sample shape: (300, 36)
Keras training class counts: {np.str_('high'): np.int64(100), np.str_('low'): np.int64(100), np.str_('medium'): np.int64(100)}


Epochs trained: 15
Best validation loss: 1.1077


## 8. Learning Curves


In [14]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history_df["loss"], label="train")
axes[0].plot(history_df["val_loss"], label="validation")
axes[0].set_title("MLP loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history_df["accuracy"], label="train")
axes[1].plot(history_df["val_accuracy"], label="validation")
axes[1].set_title("MLP accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
plt.show()


/var/folders/r_/zff9yrrx0nx1qlptw7y12jy80000gn/T/ipykernel_33393/31691853.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Final Evaluation

Macro-F1 is the main metric because rating bands can be imbalanced.


In [15]:
def keras_predict_labels(model: tf.keras.Model, X_processed: np.ndarray) -> np.ndarray:
    probabilities = model.predict(X_processed, verbose=0)
    encoded = probabilities.argmax(axis=1)
    return label_encoder.inverse_transform(encoded)

val_pred = keras_predict_labels(keras_mlp, X_val_processed)
test_pred = keras_predict_labels(keras_mlp, X_test_processed)

for split_name, y_true, y_pred in [("val", y_val, val_pred), ("test", y_test, test_pred)]:
    metrics.append(evaluate_predictions("keras_tabular_mlp", split_name, y_true, y_pred))

metrics_df = pd.DataFrame(metrics).sort_values(["split", "macro_f1"], ascending=[True, False])
metrics_df.to_csv(METRICS_PATH, index=False)
metrics_df.to_csv(MODEL_COMPARISON_PATH, index=False)
display(metrics_df.round(4))
print("Saved metrics:", METRICS_PATH.relative_to(PROJECT_ROOT))
print("Saved model comparison:", MODEL_COMPARISON_PATH.relative_to(PROJECT_ROOT))


,model,split,accuracy,macro_f1,weighted_f1
3,logistic_regression_baseline,test,0.4506,0.4292,0.4604
5,keras_tabular_mlp,test,0.3169,0.3117,0.2990
1,majority_baseline,test,0.5279,0.2303,0.3647
2,logistic_regression_baseline,val,0.4627,0.4405,0.4719
4,keras_tabular_mlp,val,0.3216,0.3179,0.3010
0,majority_baseline,val,0.5284,0.2305,0.3653


Saved metrics: reports/milestone1_tabular_mlp_metrics.csv


In [16]:
report_dict = classification_report(
    y_test,
    test_pred,
    labels=BAND_ORDER,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
report_df.to_csv(CLASSIFICATION_REPORT_PATH)
display(report_df.round(4))
print("Saved classification report:", CLASSIFICATION_REPORT_PATH.relative_to(PROJECT_ROOT))


,precision,recall,f1-score,support
low,0.3387,0.5048,0.4054,1157.0000
medium,0.2046,0.4906,0.2887,640.0000
high,0.5631,0.1533,0.2410,2009.0000
accuracy,0.3169,0.3169,0.3169,0.3169
macro avg,0.3688,0.3829,0.3117,3806.0000
weighted avg,0.4346,0.3169,0.2990,3806.0000


Saved classification report: reports/milestone1_tabular_mlp_classification_report.csv


In [17]:
cm = confusion_matrix(y_test, test_pred, labels=BAND_ORDER)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=BAND_ORDER, yticklabels=BAND_ORDER, ax=ax)
ax.set_title("Keras Tabular MLP confusion matrix - test")
ax.set_xlabel("Predicted rating band")
ax.set_ylabel("True rating band")
plt.show()

confusions = pd.DataFrame(cm, index=BAND_ORDER, columns=BAND_ORDER)
confusion_pairs = []
for true_label in BAND_ORDER:
    for pred_label in BAND_ORDER:
        if true_label != pred_label:
            confusion_pairs.append({
                "true_rating_band": true_label,
                "predicted_rating_band": pred_label,
                "count": int(confusions.loc[true_label, pred_label]),
            })
confusion_pairs = pd.DataFrame(confusion_pairs).sort_values("count", ascending=False)
display(confusion_pairs)


/var/folders/r_/zff9yrrx0nx1qlptw7y12jy80000gn/T/ipykernel_33393/3203931732.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,true_rating_band,predicted_rating_band,count
4,high,low,903
5,high,medium,798
0,low,medium,423
2,medium,low,237
1,low,high,150
3,medium,high,89


## 10. Error Analysis

Misclassified product examples help reveal which product metadata patterns the model struggles with.


In [18]:
test_predictions = product_test_df[[
    "product_id",
    "product_title",
    "price_num",
    "rating_number",
    TARGET,
    TARGET_SOURCE,
]].copy()
test_predictions["predicted_rating_band"] = test_pred

proba = keras_mlp.predict(X_test_processed, verbose=0)
for idx, label in enumerate(label_encoder.classes_):
    test_predictions[f"proba_{label}"] = proba[:, idx]

test_predictions["is_correct"] = test_predictions[TARGET].astype(str) == test_predictions["predicted_rating_band"].astype(str)
misclassified_products = test_predictions.query("not is_correct").copy()

display(misclassified_products[[
    "product_id",
    "product_title",
    "price_num",
    "rating_number",
    TARGET,
    "predicted_rating_band",
]].head(25))

print("Most common confusions:")
display(confusion_pairs.head(5))


,product_id,product_title,price_num,rating_number,rating_band,predicted_rating_band
21566,B08FZB43ZK,3D Mask Bracket for Kids，Internal Support Holder Frame Nose Breathing Smoothly - Protect Lipstick Lips - Cool Face M...,NaN,14,high,low
21567,B07KFPH3S7,Lechat Perfect Match High Gloss Top Gel Sealer + Untra-Thin Base Gel 0.25oz,NaN,25,high,low
21568,B08LYZY994,Body wave Lace Front Wigs Human Hair 4X4 Body Wave 150% Density Pre Plucked with Baby Hair Natural Black 100% Unproc...,NaN,22,high,low
21569,B0159D9ZJ4,Barber Sponge Brush Double Side 2 Different Wave Sizes 2 PC,NaN,38,high,low
21570,B00759LVR0,J. Leblanc French Pistachio Oil - 8 fl.oz. - Huile de Pistache,59.12,1,high,low
21577,B07LGZF4YV,"Beard Grooming Kit for Men Care - 100% Organic Unscented Beard Oil, Beard Shampoo, Beard Brush, Wooden Beard Comb, M...",NaN,58,high,low
21579,B00NNLWGXW,10pcs Woman Clear Acrylic Nose Bone Studs Rings Retainers Body Piercing Jewelry,NaN,71,low,medium
21580,B09NVN6WTC,Soap & Glory The Big Pink Collection Gift Set,NaN,5,high,low
21581,B017F2OH2M,Galaxy Print Nail Stickers - 3 Pack (42 Total Nail Art Wraps) Shooting Stars for Easy Nail Art and Pretty Nails That...,NaN,66,medium,low
21583,B08RDS5G49,8 Pieces African Headband Stretchy Boho Print Head Band Yoga Sports Workout Hairband Elastic Turban Headwrap Head We...,NaN,12,high,low


Most common confusions:


,true_rating_band,predicted_rating_band,count
4,high,low,903
5,high,medium,798
0,low,medium,423
2,medium,low,237
1,low,high,150


## 11. Save Model, Preprocessor, and Predictions


In [19]:
keras_mlp.save(KERAS_MODEL_PATH)
joblib.dump(preprocessor, PREPROCESSOR_PATH)
joblib.dump(label_encoder, LABEL_ENCODER_PATH)

val_predictions = product_val_df[["product_id", "product_title", "price_num", "rating_number", TARGET, TARGET_SOURCE]].copy()
val_predictions["predicted_rating_band"] = val_pred
val_proba = keras_mlp.predict(X_val_processed, verbose=0)
for idx, label in enumerate(label_encoder.classes_):
    val_predictions[f"proba_{label}"] = val_proba[:, idx]

val_predictions.to_csv(VAL_PREDICTIONS_PATH, index=False)
test_predictions.to_csv(TEST_PREDICTIONS_PATH, index=False)

print("Saved Keras model:", KERAS_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Saved preprocessor:", PREPROCESSOR_PATH.relative_to(PROJECT_ROOT))
print("Saved label encoder:", LABEL_ENCODER_PATH.relative_to(PROJECT_ROOT))
print("Saved validation predictions:", VAL_PREDICTIONS_PATH.relative_to(PROJECT_ROOT))
print("Saved test predictions:", TEST_PREDICTIONS_PATH.relative_to(PROJECT_ROOT))


Saved Keras model: models/tabular_mlp.keras
Saved preprocessor: models/tabular_mlp_preprocessor.joblib
Saved label encoder: models/tabular_mlp_label_encoder.joblib
Saved validation predictions: data/processed/tabular_mlp_predictions_val.csv
Saved test predictions: data/processed/tabular_mlp_predictions_test.csv


## 12. Baseline Comparison Check

This check is intentionally honest. If the Keras MLP does not beat Logistic Regression on validation macro-F1, we keep the result and interpret it. For this tabular metadata problem, a linear baseline can be stronger because the feature set is small, mostly numeric/count-based, and has many sparse one-hot store/category indicators. A neural network may need more examples, richer features, or tuning to outperform the simpler baseline.


In [ ]:
logreg_val_macro_f1 = metrics_df.query("model == 'logistic_regression_baseline' and split == 'val'")["macro_f1"].iloc[0]
mlp_val_macro_f1 = metrics_df.query("model == 'keras_tabular_mlp' and split == 'val'")["macro_f1"].iloc[0]
mlp_beats_logreg = mlp_val_macro_f1 > logreg_val_macro_f1

print(f"MLP beats Logistic Regression on validation macro-F1: {mlp_beats_logreg}")
print(f"Logistic Regression val macro-F1: {logreg_val_macro_f1:.4f}")
print(f"Keras MLP val macro-F1: {mlp_val_macro_f1:.4f}")


## 12. Dynamic Milestone 1 Summary


In [20]:
best_val = metrics_df.query("split == 'val'").sort_values("macro_f1", ascending=False).iloc[0]
test_mlp = metrics_df.query("split == 'test' and model == 'keras_tabular_mlp'").iloc[0]

summary = {
    "modeling_level": "product-level rows",
    "target": TARGET,
    "target_strategy": target_strategy,
    "train_products": len(product_train_df),
    "val_products": len(product_val_df),
    "test_products": len(product_test_df),
    "leakage_check": "PASS" if leakage_pass else "FAIL",
    "numeric_features": len(numeric_features),
    "categorical_features": len(categorical_features),
    "best_validation_model": best_val["model"],
    "best_validation_macro_f1": round(float(best_val["macro_f1"]), 4),
    "keras_mlp_test_macro_f1": round(float(test_mlp["macro_f1"]), 4),
    "keras_model_path": str(KERAS_MODEL_PATH.relative_to(PROJECT_ROOT)),
    "metrics_path": str(METRICS_PATH.relative_to(PROJECT_ROOT)),
    "model_comparison_path": str(MODEL_COMPARISON_PATH.relative_to(PROJECT_ROOT)),
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))


,value
modeling_level,product-level rows
target,rating_band
target_strategy,"fixed rating bands: low < 3.5, medium 3.5-4.5, high >= 4.5"
train_products,17759
val_products,3806
test_products,3806
leakage_check,PASS
numeric_features,12
categorical_features,3
best_validation_model,logistic_regression_baseline
